# wandbで出力されるDFの整形用

In [19]:
import polars as pl
from pathlib import Path

In [109]:
paths = list(Path("../dataset/").rglob("*.csv"))
path = [path for path in paths if path.stem == "cisunet_vein"][0]
path

PosixPath('../dataset/wandb/cisunet_vein.csv')

In [110]:
df = pl.read_csv(path)
df = df.select(pl.exclude("^*MIN$")).select(pl.exclude("^*MAX$"))

In [111]:
rename_cols = ["Epoch"]+[col.split("_")[0]+"-"+col.split("-")[-1] for col in df.columns if "val" in col]
df = df.rename({old_col:new_col for old_col, new_col in zip(df.columns, rename_cols)})

In [112]:
result = df.unpivot(index="Epoch", variable_name="Experiment", value_name="Dice") \
    .with_columns(
        pl.col("Experiment").map_elements(lambda x: int(x.split("-")[0][-1]), return_dtype=pl.Int64).alias("Fold"),
        pl.col("Experiment").map_elements(lambda x: x.split("-")[-1], return_dtype=pl.String).alias("Label"),
        ) \
    .select(pl.exclude("Experiment")) \
    .pivot(index=["Epoch", "Fold"],values="Dice", on="Label") \
    .with_columns(
        pl.mean_horizontal(pl.exclude(["Epoch", "Fold"])).alias("mDice")
    ) \
    .filter(pl.col("mDice")==pl.max("mDice").over(pl.col("Fold"))) \
    .sort("Fold")
result

Epoch,Fold,IVC,l_CIV,l_EIV,l_IIV,r_CIV,r_EIV,r_IIV,mDice
i64,i64,f64,f64,f64,f64,f64,f64,f64,f64
70,1,0.887024,0.788686,0.817081,0.597952,0.786412,0.815664,0.591875,0.754956
87,2,0.87973,0.795701,0.838678,0.630193,0.727029,0.820348,0.571889,0.751938
79,3,0.83153,0.81634,0.821223,0.612838,0.824488,0.834783,0.597308,0.762644
97,4,0.877897,0.836629,0.839127,0.570126,0.825893,0.814262,0.603971,0.766844
88,5,0.895935,0.819425,0.846569,0.616941,0.809569,0.848653,0.564841,0.771704


In [113]:
if "Aorta" in result.columns:
    label = "Artery"
elif "IVC" in result.columns:
    label = "Vein"
else:
    label = "RV_RA"
sort_cols = {
    "Artery":["Fold", "Epoch", "Aorta", "r_CIA", "l_CIA", "IMA", "r_EIA", "r_IIA", "l_EIA", "l_IIA", "mDice"],
    "Vein":["Fold", "Epoch", "IVC", "r_CIV", "l_CIV", "r_EIV", "r_IIV", "l_EIV", "l_IIV", "mDice"],
    "RV_RA":["Fold", "Epoch", "r_RA", "l_RA", "r_RV", "l_RV"],
}
result = result.select(sort_cols[label])
result

Fold,Epoch,IVC,r_CIV,l_CIV,r_EIV,r_IIV,l_EIV,l_IIV,mDice
i64,i64,f64,f64,f64,f64,f64,f64,f64,f64
1,70,0.887024,0.786412,0.788686,0.815664,0.591875,0.817081,0.597952,0.754956
2,87,0.87973,0.727029,0.795701,0.820348,0.571889,0.838678,0.630193,0.751938
3,79,0.83153,0.824488,0.81634,0.834783,0.597308,0.821223,0.612838,0.762644
4,97,0.877897,0.825893,0.836629,0.814262,0.603971,0.839127,0.570126,0.766844
5,88,0.895935,0.809569,0.819425,0.848653,0.564841,0.846569,0.616941,0.771704
